# CDSS AI Prediction Score Calibration — 3개 질환

뇌출혈·뇌동맥류·뇌경색 모델 ID를 선택하면 validation probability와 실제 flip TTA를 사용해 case-level reliability score를 학습한다. 출력 점수는 “이 case의 최종 분할이 validation 기준상 acceptable일 확률 × 100”이며 voxel probability와 별개다.

주의: 질환별 acceptable threshold는 현재 **임시 engineering default**다. 임상 승인 전 배포하면 안 된다.

In [ ]:
from pathlib import Path
import sys
import json,os,signal,subprocess,time,yaml
import pandas as pd
from IPython.display import display,clear_output
REPO=Path('/home/stu03/workspace/model'); sys.path.insert(0,str(REPO/'src'))
from brainon_ai.calibration.registry import load_registry,resolve_model
REPO=Path('/home/stu03/workspace/model'); PYTHON=REPO/'.venv/bin/python'; REGISTRY=REPO/'configs/calibration/models.yaml'
models=load_registry(REGISTRY)
try:
 import ipywidgets as widgets
 MODEL=widgets.Dropdown(options=[(f"{k} — {v['task']}",k) for k,v in sorted(models.items())],description='Model ID:'); display(MODEL)
except ImportError:
 MODEL=None
MODEL_ID='I-S40'  # dropdown이 없으면 이 값을 수정

## 선택 모델 및 준비 상태

In [ ]:
selected=MODEL.value if MODEL is not None else MODEL_ID
m=resolve_model(selected,REGISTRY,REPO)
pred=Path(m['prediction_dir']); tta=[Path(x) for x in m['tta_prediction_dirs']]
status={'model_id':selected,'task':m['task'],'model_version':m['version'],'probability_maps':len(list(pred.glob('*.npz'))) if pred.is_dir() else 0,'tta_maps':[len(list(x.glob('*.npz'))) if x.is_dir() else 0 for x in tta],'prediction_dir':str(pred),'tta_dirs':[str(x) for x in tta]}; display(pd.DataFrame([status])); print(yaml.safe_dump(yaml.safe_load((REPO/f"configs/calibration/tasks/{m['task']}.yaml").read_text()),sort_keys=False,allow_unicode=True))

## 실행

원본 확률맵이 없으면 생성하고, 실제 단일 flip TTA를 수행한 뒤 calibration을 fitting한다. 기존 확률맵은 덮어쓰지 않는다. `ALLOW_PROVISIONAL_TARGET=True`는 임시 target 기준 사용을 명시적으로 승인하는 개발 단계 옵션이다.

In [ ]:
ALLOW_PROVISIONAL_TARGET=True
RUN_DIR=REPO/'runs/calibration_jobs'/selected; RUN_DIR.mkdir(parents=True,exist_ok=True); LOG=RUN_DIR/'run.log'; PID=RUN_DIR/'run.pid'
cmd=[str(PYTHON),str(REPO/'scripts/calibration/run_model_calibration.py'),'--model-id',selected,'--registry',str(REGISTRY)]
if ALLOW_PROVISIONAL_TARGET: cmd.append('--allow-provisional-target')
handle=LOG.open('a',buffering=1); proc=subprocess.Popen(cmd,cwd=REPO,env={**os.environ,'PYTHONPATH':str(REPO/'src')},stdout=handle,stderr=subprocess.STDOUT,start_new_session=True); PID.write_text(str(proc.pid)); print('started',proc.pid,cmd)

## 실시간 로그

In [ ]:
def alive(pid):
 try: os.kill(pid,0); return True
 except ProcessLookupError: return False
pid=int(PID.read_text())
try:
 while True:
  clear_output(wait=True); print(time.strftime('Updated %Y-%m-%d %H:%M:%S')); print(''.join(LOG.read_text(errors='replace').splitlines(True)[-30:]) if LOG.is_file() else 'waiting')
  if not alive(pid): print('작업 종료'); break
  time.sleep(10)
except KeyboardInterrupt: print('모니터만 중단했습니다. calibration 작업은 계속됩니다.')

## 결과 확인

In [ ]:
m=resolve_model(selected,REGISTRY,REPO); ART=REPO/f"artifacts/calibration/{m['task']}/{selected}/{m['version']}"
if not (ART/'calibration.json').is_file(): print('결과가 아직 없습니다:',ART)
else:
 calibration=json.loads((ART/'calibration.json').read_text()); display(pd.DataFrame(calibration['features'])); print(json.dumps(calibration['metrics'],indent=2)); display(pd.read_csv(ART/'calibration_predictions.csv').head()); display(pd.read_csv(ART/'calibration_curve.csv'))

## 단일 case feature/score 계산

In [ ]:
from brainon_ai.calibration.features import extract_features,foreground_probability
from brainon_ai.calibration.scoring import calculate_prediction_score
case_id=next(Path(m['prediction_dir']).glob('*.npz')).stem
features=extract_features(foreground_probability(Path(m['prediction_dir'])/f'{case_id}.npz'),float(m['threshold']),[foreground_probability(Path(d)/f'{case_id}.npz') for d in m['tta_prediction_dirs']])
score=calculate_prediction_score(features,calibration,expected_model_version=m['version']); print(case_id,features,'score=',score)